In [ ]:
# The first section imports the required libraries, identifies the project folder, and loads train.csv and test.csv.
# The original CSV files are only read and are not modified.
#
# The second section creates the target variable from Closed Date and Created Date, extracts basic temporal features
# from Created Date, converts known numeric columns, and removes leakage or identifier columns.
#
# The third section handles missing values. Columns with too many missing values are dropped, numerical missing values
# are replaced with medians calculated from the training data, and categorical missing values are replaced with UNKNOWN.
#
# The fourth section creates derived temporal features, including cyclic transformations for hour, day of week, and month.
# It also adds indicators for business-hour requests and night-time requests.
#
# The fifth section creates geographic features. It adds explicit Borough indicators, groups Incident Zip into broader
# zip-code areas, calculates the approximate distance from the centre of New York City using latitude and longitude,
# and creates a latitude-longitude interaction feature.
#
# The sixth section applies count encoding to important categorical and geographic variables, including Agency,
# Agency Name, Problem Type, Borough, Incident Zip, and incident_zip_group. The frequency values are learned only
# from the training data and then applied to the test data to avoid data leakage.
#
# The seventh section creates interaction features, such as weekend-night requests, business-hour weekday requests,
# and the interaction between Borough frequency and weekend requests.
#
# The eighth section converts the remaining categorical variables into numerical format using One-Hot Encoding for
# low-cardinality columns and Label Encoding for high-cardinality columns.
#
# The ninth section creates the final matrices for modelling: X_train, y_train, and X_test. It also checks that there
# are no leakage columns, missing values, or non-numerical columns, and that train and test have the same features.
#
# The tenth section displays useful checks and previews, including the engineered features, target distribution,
# feature quality information, summary statistics, and a small train/test comparison.

In [23]:
# data load

from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()

if not (project_root / "data" / "train.csv").exists():
    project_root = project_root.parent

data_dir = project_root / "data"

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

date_format = "%m/%d/%Y %I:%M:%S %p"

missing_drop_threshold = 0.90
max_onehot_cardinality = 15
unknown_value = "UNKNOWN"

print("train shape:", train.shape)
print("test shape:", test.shape)

train shape: (110930, 41)
test shape: (27733, 40)


In [24]:
# 2) target, temporal features, leakage removal

def add_target(df):
    df = df.copy()

    created = pd.to_datetime(
        df["Created Date"],
        format=date_format,
        errors="coerce"
    )

    closed = pd.to_datetime(
        df["Closed Date"],
        format=date_format,
        errors="coerce"
    )

    hours_to_close = (closed - created).dt.total_seconds() / 3600

    df["target"] = (
        (hours_to_close >= 0)
        & (hours_to_close <= 24)
    ).astype(int)

    return df


def add_temporal_features(df):
    df = df.copy()

    created = pd.to_datetime(
        df["Created Date"],
        format=date_format,
        errors="coerce"
    )

    hour = created.dt.hour

    df["created_hour"] = hour
    df["created_day_of_week"] = created.dt.dayofweek
    df["created_month"] = created.dt.month
    df["created_is_weekend"] = created.dt.dayofweek.isin([5, 6]).astype(int)

    df["created_part_of_day"] = np.select(
        [
            hour.between(0, 5),
            hour.between(6, 11),
            hour.between(12, 17),
            hour.between(18, 23)
        ],
        [
            "night",
            "morning",
            "afternoon",
            "evening"
        ],
        default=unknown_value
    )

    return df


def convert_known_numeric_columns(df):
    df = df.copy()

    numeric_cols = [
        "Incident Zip",
        "Council District",
        "BBL",
        "X Coordinate (State Plane)",
        "Y Coordinate (State Plane)",
        "Latitude",
        "Longitude"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(",", ".", regex=False)
                .str.strip()
            )

            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def remove_leakage_and_ids(df):
    df = df.copy()

    cols_to_drop = [
        "Closed Date",
        "Created Date",
        "Unique Key",
        "Unnamed: 0",
        "Location"
    ]

    existing_cols = [
        col for col in cols_to_drop
        if col in df.columns
    ]

    return df.drop(columns=existing_cols)


train_processed = add_target(train)
train_processed = add_temporal_features(train_processed)
train_processed = convert_known_numeric_columns(train_processed)
train_processed = remove_leakage_and_ids(train_processed)

test_processed = add_temporal_features(test)
test_processed = convert_known_numeric_columns(test_processed)
test_processed = remove_leakage_and_ids(test_processed)

print(train_processed["target"].value_counts())
print(train_processed["target"].value_counts(normalize=True).round(4))

print("train processed shape:", train_processed.shape)
print("test processed shape:", test_processed.shape)

target
1    68085
0    42845
Name: count, dtype: int64
target
1    0.6138
0    0.3862
Name: proportion, dtype: float64
train processed shape: (110930, 42)
test processed shape: (27733, 41)


In [25]:
# 3) missing values

def fit_missing_value_params(df):
    missing_fraction = df.isna().mean()

    dropped_cols = [
        col
        for col in missing_fraction[
            missing_fraction > missing_drop_threshold
        ].index
        if col != "target"
    ]

    remaining = df.drop(columns=dropped_cols, errors="ignore")

    numeric_data = (
        remaining
        .select_dtypes(include="number")
        .drop(columns=["target"], errors="ignore")
    )

    numeric_medians = numeric_data.median().to_dict()

    return {
        "dropped_cols": dropped_cols,
        "numeric_medians": numeric_medians,
        "categorical_fill_value": unknown_value
    }


def apply_missing_value_params(df, params):
    df = df.copy()

    df = df.drop(columns=params["dropped_cols"], errors="ignore")

    for col, median in params["numeric_medians"].items():
        if col in df.columns:
            df[col] = df[col].fillna(median)

    categorical_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in categorical_cols:
        df[col] = (
            df[col]
            .fillna(params["categorical_fill_value"])
            .astype(str)
        )

    return df


missing_params = fit_missing_value_params(train_processed)

train_processed = apply_missing_value_params(train_processed, missing_params)
test_processed = apply_missing_value_params(test_processed, missing_params)

print("dropped columns:")
print(missing_params["dropped_cols"])

print("train missing values:", train_processed.isna().sum().sum())
print("test missing values:", test_processed.isna().sum().sum())

dropped columns:
['Facility Type', 'X Coordinate (State Plane)', 'Y Coordinate (State Plane)', 'Vehicle Type', 'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name', 'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment']
train missing values: 0
test missing values: 0


In [26]:
# 4) Derived temporal features

def add_derived_temporal_features(df):
    df = df.copy()

    df["created_hour_sin"] = np.sin(2 * np.pi * df["created_hour"] / 24)
    df["created_hour_cos"] = np.cos(2 * np.pi * df["created_hour"] / 24)

    df["created_day_sin"] = np.sin(
        2 * np.pi * df["created_day_of_week"] / 7
    )

    df["created_day_cos"] = np.cos(
        2 * np.pi * df["created_day_of_week"] / 7
    )

    df["created_month_sin"] = np.sin(
        2 * np.pi * df["created_month"] / 12
    )

    df["created_month_cos"] = np.cos(
        2 * np.pi * df["created_month"] / 12
    )

    df["created_is_business_hour"] = (
        df["created_hour"].between(9, 17)
    ).astype(int)

    df["created_is_night"] = (
        df["created_hour"].between(0, 5)
    ).astype(int)

    return df


train_processed = add_derived_temporal_features(train_processed)
test_processed = add_derived_temporal_features(test_processed)

temporal_features = [
    "created_hour_sin",
    "created_hour_cos",
    "created_day_sin",
    "created_day_cos",
    "created_month_sin",
    "created_month_cos",
    "created_is_business_hour",
    "created_is_night"
]

print("temporal features added:", temporal_features)

temporal features added: ['created_hour_sin', 'created_hour_cos', 'created_day_sin', 'created_day_cos', 'created_month_sin', 'created_month_cos', 'created_is_business_hour', 'created_is_night']


In [27]:
# 5) geographic features

def add_geographic_features(df):
    df = df.copy()

    if "Borough" in df.columns:
        borough = df["Borough"].astype(str).str.upper()

        df["is_manhattan"] = (borough == "MANHATTAN").astype(int)
        df["is_brooklyn"] = (borough == "BROOKLYN").astype(int)
        df["is_queens"] = (borough == "QUEENS").astype(int)
        df["is_bronx"] = (borough == "BRONX").astype(int)
        df["is_staten_island"] = (borough == "STATEN ISLAND").astype(int)

    if "Incident Zip" in df.columns:
        df["incident_zip_group"] = (
            df["Incident Zip"] // 100
        ).astype(int).astype(str)

    if "Latitude" in df.columns and "Longitude" in df.columns:
        nyc_latitude = 40.7128
        nyc_longitude = -74.0060

        df["distance_from_nyc_center"] = np.sqrt(
            (df["Latitude"] - nyc_latitude) ** 2
            + (df["Longitude"] - nyc_longitude) ** 2
        )

        df["latitude_longitude_interaction"] = (
            df["Latitude"] * df["Longitude"]
        )

    return df


train_processed = add_geographic_features(train_processed)
test_processed = add_geographic_features(test_processed)

geographic_features = [
    col
    for col in [
        "is_manhattan",
        "is_brooklyn",
        "is_queens",
        "is_bronx",
        "is_staten_island",
        "incident_zip_group",
        "distance_from_nyc_center",
        "latitude_longitude_interaction"
    ]
    if col in train_processed.columns
]

print("geographic features added:", geographic_features)

geographic features added: ['is_manhattan', 'is_brooklyn', 'is_queens', 'is_bronx', 'is_staten_island', 'incident_zip_group', 'distance_from_nyc_center', 'latitude_longitude_interaction']


In [28]:
# 6) count encoding

def fit_count_encoding(df, columns):
    count_maps = {}

    for col in columns:
        if col in df.columns:
            count_maps[col] = (
                df[col]
                .astype(str)
                .value_counts(normalize=True)
                .to_dict()
            )

    return count_maps


def apply_count_encoding(df, count_maps):
    df = df.copy()

    for col, mapping in count_maps.items():
        if col in df.columns:
            df[f"{col}_frequency"] = (
                df[col]
                .astype(str)
                .map(mapping)
                .fillna(0)
            )

    return df


count_encoding_cols = [
    "Agency",
    "Agency Name",
    "Problem (formerly Complaint Type)",
    "Borough",
    "Incident Zip",
    "incident_zip_group"
]

count_maps = fit_count_encoding(
    train_processed,
    count_encoding_cols
)

train_processed = apply_count_encoding(
    train_processed,
    count_maps
)

test_processed = apply_count_encoding(
    test_processed,
    count_maps
)

count_features = [
    f"{col}_frequency"
    for col in count_encoding_cols
    if f"{col}_frequency" in train_processed.columns
]

print("count-encoded features added:", count_features)

count-encoded features added: ['Agency_frequency', 'Agency Name_frequency', 'Problem (formerly Complaint Type)_frequency', 'Borough_frequency', 'Incident Zip_frequency', 'incident_zip_group_frequency']


In [29]:
# 7) interaction features

def add_interaction_features(df):
    df = df.copy()

    df["weekend_night_interaction"] = (
        df["created_is_weekend"] * df["created_is_night"]
    )

    df["business_hour_weekday_interaction"] = (
        df["created_is_business_hour"] * (1 - df["created_is_weekend"])
    )

    if "Borough_frequency" in df.columns:
        df["borough_weekend_interaction"] = (
            df["Borough_frequency"] * df["created_is_weekend"]
        )

    return df


train_processed = add_interaction_features(train_processed)
test_processed = add_interaction_features(test_processed)

interaction_features = [
    col
    for col in [
        "weekend_night_interaction",
        "business_hour_weekday_interaction",
        "borough_weekend_interaction"
    ]
    if col in train_processed.columns
]

print("interaction features added:", interaction_features)

interaction features added: ['weekend_night_interaction', 'business_hour_weekday_interaction', 'borough_weekend_interaction']


In [30]:
# 8) categorical encoding

def fit_categorical_encoding_params(df):
    categorical_cols = (
        df
        .select_dtypes(include=["object", "category"])
        .columns
        .tolist()
    )

    categorical_cols = [
        col
        for col in categorical_cols
        if col != "target"
    ]

    onehot_cols = []
    label_maps = {}

    for col in categorical_cols:
        n_unique = df[col].nunique(dropna=False)

        if n_unique <= max_onehot_cardinality:
            onehot_cols.append(col)
        else:
            categories = sorted(df[col].astype(str).unique().tolist())

            mapping = {
                category: index
                for index, category in enumerate(categories)
            }

            mapping["__UNKNOWN__"] = len(mapping)
            label_maps[col] = mapping

    return {
        "onehot_cols": onehot_cols,
        "label_maps": label_maps
    }


def apply_categorical_encoding(df, params):
    df = df.copy()

    for col, mapping in params["label_maps"].items():
        if col in df.columns:
            unknown_code = mapping["__UNKNOWN__"]

            df[col] = (
                df[col]
                .astype(str)
                .map(mapping)
                .fillna(unknown_code)
                .astype(int)
            )

    existing_onehot_cols = [
        col
        for col in params["onehot_cols"]
        if col in df.columns
    ]

    df = pd.get_dummies(
        df,
        columns=existing_onehot_cols,
        dtype=int
    )

    return df


encoding_params = fit_categorical_encoding_params(train_processed)

train_encoded = apply_categorical_encoding(
    train_processed,
    encoding_params
)

test_encoded = apply_categorical_encoding(
    test_processed,
    encoding_params
)

print("one-hot columns:")
print(encoding_params["onehot_cols"])

print("label-encoded columns:")
print(list(encoding_params["label_maps"].keys()))

print("train shape after encoding:", train_encoded.shape)
print("test shape after encoding:", test_encoded.shape)

one-hot columns:
['Agency', 'Agency Name', 'Address Type', 'Borough', 'Open Data Channel Type', 'Park Facility Name', 'Park Borough', 'created_part_of_day', 'incident_zip_group']
label-encoded columns:
['Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Additional Details', 'Location Type', 'Incident Address', 'Street Name', 'Cross Street 1', 'Cross Street 2', 'Intersection Street 1', 'Intersection Street 2', 'City', 'Landmark', 'Community Board', 'Police Precinct']
train shape after encoding: (110930, 125)
test shape after encoding: (27733, 120)


In [31]:
# 9) final matrices and checks

def create_final_matrices(train_df, test_df):
    feature_cols = [
        col
        for col in train_df.columns
        if col != "target"
    ]

    X_train = train_df[feature_cols].copy()
    y_train = train_df["target"].copy()

    X_test = test_df.reindex(
        columns=feature_cols,
        fill_value=0
    ).copy()

    return X_train, y_train, X_test


X_train, y_train, X_test = create_final_matrices(
    train_encoded,
    test_encoded
)

assert "target" not in X_train.columns
assert "target" not in X_test.columns
assert "Closed Date" not in X_train.columns
assert "Closed Date" not in X_test.columns
assert X_train.shape[1] == X_test.shape[1]
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0
assert len(X_train.select_dtypes(exclude="number").columns) == 0
assert len(X_test.select_dtypes(exclude="number").columns) == 0

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (110930, 124)
y_train shape: (110930,)
X_test shape: (27733, 124)


In [32]:
# 10) final visualisation

engineered_features = (
    temporal_features
    + geographic_features
    + count_features
    + interaction_features
)

engineered_features_in_train_processed = [
    col for col in engineered_features
    if col in train_processed.columns
]

engineered_features_in_X_train = [
    col for col in engineered_features
    if col in X_train.columns
]

encoded_engineered_features = [
    col for col in X_train.columns
    if any(
        col.startswith(feature + "_")
        for feature in engineered_features
    )
]

all_engineered_features_in_X_train = (
    engineered_features_in_X_train
    + encoded_engineered_features
)

print("Cell 10.1 — general feature engineering summary")
print("engineered features before encoding:", len(engineered_features))
print("engineered features visible before categorical encoding:", len(engineered_features_in_train_processed))
print("engineered features visible after categorical encoding:", len(all_engineered_features_in_X_train))
print("final X_train shape:", X_train.shape)
print("final X_test shape:", X_test.shape)

print("\nCell 10.2 — preview of engineered features before categorical encoding")

engineered_preview = (
    train_processed[engineered_features_in_train_processed]
    .head()
    .T
    .rename(columns={
        0: "row_0",
        1: "row_1",
        2: "row_2",
        3: "row_3",
        4: "row_4"
    })
)

display(engineered_preview)

print("\nCell 10.3 — target distribution used for supervised learning")

target_summary = (
    y_train
    .value_counts()
    .rename_axis("target")
    .reset_index(name="count")
)

target_summary["percentage"] = (
    target_summary["count"] / len(y_train) * 100
).round(2)

display(target_summary)

print("\nCell 10.4 — feature quality check after preprocessing and encoding")

feature_quality = pd.DataFrame({
    "missing_values": X_train.isna().sum(),
    "unique_values": X_train.nunique(),
    "dtype": X_train.dtypes.astype(str)
})

feature_quality = (
    feature_quality
    .sort_values("unique_values", ascending=False)
    .head(20)
)

display(feature_quality)

print("\nCell 10.5 — statistical summary of engineered numeric features")

numeric_engineered_features = [
    col for col in all_engineered_features_in_X_train
    if col in X_train.columns
]

engineered_feature_summary = (
    X_train[numeric_engineered_features]
    .describe()
    .T[["mean", "std", "min", "max"]]
    .round(4)
)

display(engineered_feature_summary)

print("\nCell 10.6 — train/test comparison for selected engineered features")

sample_columns = numeric_engineered_features[:10]

comparison_preview = pd.concat(
    [
        X_train[sample_columns].head().add_prefix("train_"),
        X_test[sample_columns].head().add_prefix("test_")
    ],
    axis=1
)

display(comparison_preview)

Cell 10.1 — general feature engineering summary
engineered features before encoding: 25
engineered features visible before categorical encoding: 25
engineered features visible after categorical encoding: 40
final X_train shape: (110930, 124)
final X_test shape: (27733, 124)

Cell 10.2 — preview of engineered features before categorical encoding


,row_0,row_1,row_2,row_3,row_4
created_hour_sin,0.866025,-0.707107,0.0,0.0,0.707107
created_hour_cos,-0.5,-0.707107,-1.0,-1.0,-0.707107
created_day_sin,0.781831,0.433884,-0.781831,-0.433884,0.433884
created_day_cos,0.62349,-0.900969,0.62349,-0.900969,-0.900969
created_month_sin,0.866025,0.866025,0.866025,0.866025,0.866025
created_month_cos,-0.5,-0.5,-0.5,-0.5,-0.5
created_is_business_hour,0,1,1,1,1
created_is_night,0,0,0,0,0
is_manhattan,1,1,0,0,1
is_brooklyn,0,0,1,0,0



Cell 10.3 — target distribution used for supervised learning


,target,count,percentage
0,1,68085,61.38
1,0,42845,38.62



Cell 10.4 — feature quality check after preprocessing and encoding


,missing_values,unique_values,dtype
latitude_longitude_interaction,0,55256,float64
distance_from_nyc_center,0,55256,float64
Longitude,0,55255,float64
Latitude,0,55255,float64
Incident Address,0,52949,int64
BBL,0,43815,float64
Cross Street 1,0,7200,int64
Cross Street 2,0,7078,int64
Street Name,0,6211,int64
Intersection Street 2,0,5149,int64



Cell 10.5 — statistical summary of engineered numeric features


,mean,std,min,max
created_hour_sin,-0.1572,0.6672,-1.0000,1.0000
created_hour_cos,-0.1900,0.7028,-1.0000,1.0000
created_day_sin,0.0483,0.7003,-0.9749,0.9749
created_day_cos,0.0202,0.7119,-0.9010,1.0000
created_month_sin,0.8660,0.0000,0.8660,0.8660
created_month_cos,-0.5000,0.0000,-0.5000,-0.5000
created_is_business_hour,0.5089,0.4999,0.0000,1.0000
created_is_night,0.1044,0.3058,0.0000,1.0000
is_manhattan,0.2032,0.4024,0.0000,1.0000
is_brooklyn,0.2974,0.4571,0.0000,1.0000



Cell 10.6 — train/test comparison for selected engineered features


,train_created_hour_sin,train_created_hour_cos,train_created_day_sin,train_created_day_cos,train_created_month_sin,train_created_month_cos,train_created_is_business_hour,train_created_is_night,train_is_manhattan,train_is_brooklyn,test_created_hour_sin,test_created_hour_cos,test_created_day_sin,test_created_day_cos,test_created_month_sin,test_created_month_cos,test_created_is_business_hour,test_created_is_night,test_is_manhattan,test_is_brooklyn
0,8.660254e-01,-0.500000,0.781831,0.623490,0.866025,-0.5,0,0,1,0,0.258819,-0.965926,0.433884,-0.900969,0.866025,-0.5,1,0,0,1
1,-7.071068e-01,-0.707107,0.433884,-0.900969,0.866025,-0.5,1,0,1,0,0.500000,-0.866025,-0.781831,0.623490,0.866025,-0.5,1,0,0,0
2,1.224647e-16,-1.000000,-0.781831,0.623490,0.866025,-0.5,1,0,0,1,0.866025,-0.500000,-0.433884,-0.900969,0.866025,-0.5,0,0,0,0
3,1.224647e-16,-1.000000,-0.433884,-0.900969,0.866025,-0.5,1,0,0,0,0.707107,-0.707107,0.000000,1.000000,0.866025,-0.5,1,0,0,1
4,7.071068e-01,-0.707107,0.433884,-0.900969,0.866025,-0.5,1,0,1,0,0.500000,-0.866025,0.781831,0.623490,0.866025,-0.5,1,0,1,0
